# Sequence Models to BERT

**Course:** [Natural Language Processing](https://ml-viz.vercel.app/courses/nlp/03-sequence-models-to-bert)

This notebook visualizes Bahdanau attention alignment in seq2seq models, implements a simplified Masked Language Modeling (MLM) objective, and demonstrates fine-tuning heads on top of frozen Transformer representations.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Bahdanau Attention — alignment visualization

Attention computes alignment weights α_{t,s}: how much each source position s contributes to generating target token t.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

# Simulate Bahdanau attention for a simple English→French translation
# Source: "The cat sat on the mat"
# Target: "Le chat s'est assis sur le tapis"
source = ['The', 'cat', 'sat', 'on', 'the', 'mat']
target = ['Le', 'chat', 's\'est', 'assis', 'sur', 'le', 'tapis']

# Simulated attention weights (rows=target, cols=source)
# Hand-crafted to approximate plausible word alignments
alpha = np.array([
    [0.85, 0.05, 0.02, 0.02, 0.05, 0.01],  # Le ← The
    [0.05, 0.88, 0.03, 0.01, 0.02, 0.01],  # chat ← cat
    [0.03, 0.05, 0.80, 0.07, 0.03, 0.02],  # s'est ← sat
    [0.02, 0.04, 0.82, 0.06, 0.04, 0.02],  # assis ← sat
    [0.02, 0.02, 0.03, 0.88, 0.03, 0.02],  # sur ← on
    [0.04, 0.02, 0.02, 0.02, 0.85, 0.05],  # le ← the
    [0.02, 0.03, 0.02, 0.02, 0.03, 0.88],  # tapis ← mat
])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(alpha, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(source)))
ax.set_xticklabels(source, fontsize=11)
ax.set_yticks(range(len(target)))
ax.set_yticklabels(target, fontsize=11)
ax.set_xlabel('Source (English)', fontsize=11)
ax.set_ylabel('Target (French)', fontsize=11)
ax.set_title('Bahdanau attention alignment — The cat sat on the mat', fontsize=11)
plt.colorbar(im, ax=ax, label='Attention weight α')
plt.tight_layout()
plt.show()

## The bottleneck problem

Without attention, seq2seq quality degrades sharply for long sequences because all information must be compressed into a single fixed-size vector.

In [ ]:
# Empirical BLEU-like quality degradation curve (from Bahdanau et al. 2015 data)
lengths = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60])
bleu_without_attn = np.array([26, 24, 22, 20, 17, 14, 11, 9, 7, 5, 4])
bleu_with_attn = np.array([27, 26, 25, 25, 24, 24, 23, 23, 22, 22, 21])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lengths, bleu_without_attn, 'o-', color='#ef4444', label='Seq2seq (no attention)', linewidth=2)
ax.plot(lengths, bleu_with_attn, 's-', color='#2dd4bf', label='Seq2seq + Bahdanau attention', linewidth=2)
ax.fill_between(lengths, bleu_without_attn, bleu_with_attn, alpha=0.15, color='#6366f1')
ax.set_xlabel('Source sentence length (words)', fontsize=11)
ax.set_ylabel('BLEU score (translation quality)', fontsize=11)
ax.set_title('Attention fixes the quality degradation for long sequences', fontsize=11)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## BERT MLM: Masked Language Modeling

BERT masks 15% of tokens and predicts the original. Of those 15%: 80% get `[MASK]`, 10% random token, 10% unchanged.

In [ ]:
def apply_mlm_masking(tokens, mask_prob=0.15, rng=None):
    """
    Apply BERT's MLM masking strategy.
    Returns (masked_tokens, labels) where labels[i] is the original token
    if position i was chosen for masking, else None.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    vocab = list(set(tokens))
    masked = tokens.copy()
    labels = [None] * len(tokens)
    
    for i, tok in enumerate(tokens):
        if rng.random() < mask_prob:
            labels[i] = tok  # remember original
            r = rng.random()
            if r < 0.80:
                masked[i] = '[MASK]'
            elif r < 0.90:
                masked[i] = rng.choice(vocab)  # random token
            # else: leave unchanged (10%)
    return masked, labels

sentence = '[CLS] the cat sat on the mat [SEP]'.split()
masked, labels = apply_mlm_masking(sentence)

print("Original:")
print(' '.join(f'{t:>8}' for t in sentence))
print("\nMasked input:")
print(' '.join(f'{t:>8}' for t in masked))
print("\nLabels (None = not masked):")
print(' '.join(f'{str(l):>8}' for l in labels))

In [ ]:
# Visualize masking distribution across many applications
n_trials = 10000
n_tokens = 128
rng = np.random.default_rng(99)

mask_counts = []
for _ in range(n_trials):
    selected = rng.random(n_tokens) < 0.15
    mask_counts.append(selected.sum())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mask_counts, bins=range(5, 35), color='#6366f1', alpha=0.8, edgecolor='#4f46e5')
ax.axvline(np.mean(mask_counts), color='#f97316', linestyle='--', linewidth=2,
           label=f'Mean = {np.mean(mask_counts):.1f} = 15% × {n_tokens}')
ax.set_xlabel('Number of masked tokens per sequence (length=128)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('MLM masking follows a binomial distribution', fontsize=11)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
plt.tight_layout()
plt.show()

## BERT Fine-tuning

Pre-trained BERT contextual representations can be used for multiple tasks by adding a thin task-specific head.

In [ ]:
# Simulate "frozen BERT" features + linear head for sentiment classification
# In practice you'd use HuggingFace transformers; here we show the concept

def simulate_bert_cls(texts, hidden_dim=64, rng=None):
    """
    Return fake [CLS] token embeddings — simulating what BERT would produce.
    Positive texts get embeddings biased in one direction, negative in another.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    embeddings = []
    for text, label in texts:
        base = rng.standard_normal(hidden_dim)
        signal = np.ones(hidden_dim) if label == 1 else -np.ones(hidden_dim)
        # BERT representations: strong signal + noise
        emb = 0.3 * signal + 0.7 * base
        embeddings.append(emb)
    return np.array(embeddings)

# Simulated dataset
train_data = [
    ("This film is excellent", 1), ("Wonderful performance", 1),
    ("Amazing story", 1), ("Loved every minute", 1), ("Brilliant direction", 1),
    ("Terrible movie", 0), ("Complete waste of time", 0),
    ("Boring and predictable", 0), ("Awful acting", 0), ("Disappointing", 0),
]

X_train = simulate_bert_cls(train_data)
y_train = np.array([label for _, label in train_data])

print(f"Feature matrix shape: {X_train.shape}")
print(f"Labels: {y_train}")

In [ ]:
# Train a linear classifier on [CLS] features
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -20, 20)))

def train_linear_head(X, y, lr=0.01, epochs=200):
    W = np.zeros(X.shape[1])
    b = 0.0
    losses = []
    for _ in range(epochs):
        logits = X @ W + b
        preds = sigmoid(logits)
        loss = -np.mean(y * np.log(preds + 1e-8) + (1-y) * np.log(1-preds + 1e-8))
        losses.append(loss)
        dW = X.T @ (preds - y) / len(y)
        db = np.mean(preds - y)
        W -= lr * dW
        b -= lr * db
    return W, b, losses

W_head, b_head, losses = train_linear_head(X_train, y_train)

preds = (sigmoid(X_train @ W_head + b_head) > 0.5).astype(int)
acc = (preds == y_train).mean()
print(f"Training accuracy: {acc:.1%}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, color='#6366f1', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Linear head on frozen BERT [CLS] features', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1: Compute attention context vector

Given encoder hidden states and attention weights, compute the context vector c_t = Σ α_{t,s} · h_s.

In [ ]:
def compute_context_vector(encoder_hidden_states, attention_weights):
    """
    Compute the attention context vector.
    
    Args:
        encoder_hidden_states: np.ndarray of shape (S, D)
                               S = source sequence length, D = hidden dimension
        attention_weights: np.ndarray of shape (S,)
                           Should sum to 1 (already softmax-normalized)
    Returns:
        np.ndarray of shape (D,) — the weighted sum context vector
    """
    # TODO(you): compute the weighted sum c_t = sum_s alpha_s * h_s
    pass


# Test
S, D = 6, 4
H = np.random.randn(S, D)
alpha = softmax(np.random.randn(S))
c = compute_context_vector(H, alpha)
print(f"Context vector shape: {c.shape if c is not None else None}")
print(f"Context vector: {c}")

In [ ]:
S, D = 6, 4
H = np.array([[float(i + j) for j in range(D)] for i in range(S)])
alpha = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0])  # full weight on last state
c = compute_context_vector(H, alpha)
assert c is not None, "Should return a vector"
assert c.shape == (D,), f"Expected shape ({D},), got {c.shape}"
np.testing.assert_allclose(c, H[-1], rtol=1e-5,
    err_msg="With weight=1 on last state, context should equal that state")
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compute_context_vector(encoder_hidden_states, attention_weights):
    # Weighted sum: (S,) x (S, D) → (D,)
    return attention_weights @ encoder_hidden_states
```
</details>

### Exercise 2: Implement MLM masking statistics

BERT selects 15% of tokens. Of those selected: 80% become `[MASK]`, 10% random, 10% unchanged.

Implement a function that, given a sequence of tokens, returns the counts of each masking action applied.

In [ ]:
def count_mlm_actions(tokens, mask_prob=0.15, rng_seed=42):
    """
    Apply BERT MLM masking and return action counts.
    
    Args:
        tokens: list of str
        mask_prob: float, probability of selecting a token for masking
        rng_seed: int
    Returns:
        dict with keys 'masked', 'random', 'unchanged', 'not_selected'
        where values are counts of tokens in each category
    """
    rng = np.random.default_rng(rng_seed)
    counts = {'masked': 0, 'random': 0, 'unchanged': 0, 'not_selected': 0}
    
    # TODO(you): for each token, determine if it's selected (prob mask_prob)
    # If selected: 80% → 'masked', 10% → 'random', 10% → 'unchanged'
    # If not selected: 'not_selected'
    
    return counts


long_seq = ['word'] * 200
result = count_mlm_actions(long_seq)
print("MLM action counts:", result)
print(f"Total: {sum(result.values())} (should be {len(long_seq)})")

In [ ]:
result = count_mlm_actions(['word'] * 1000, rng_seed=0)
total = sum(result.values())
assert total == 1000, f"Counts should sum to 1000, got {total}"
selected = result['masked'] + result['random'] + result['unchanged']
# ~15% should be selected (±5% for randomness)
assert 100 <= selected <= 200, f"Expected ~150 selected tokens, got {selected}"
# Of selected: ~80% masked
assert result['masked'] > result['random'], "Masked count should dominate"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def count_mlm_actions(tokens, mask_prob=0.15, rng_seed=42):
    rng = np.random.default_rng(rng_seed)
    counts = {'masked': 0, 'random': 0, 'unchanged': 0, 'not_selected': 0}
    for tok in tokens:
        if rng.random() < mask_prob:
            r = rng.random()
            if r < 0.80:
                counts['masked'] += 1
            elif r < 0.90:
                counts['random'] += 1
            else:
                counts['unchanged'] += 1
        else:
            counts['not_selected'] += 1
    return counts
```
</details>